# Fase 2: Exploración, Limpieza Interanual, Transformación y Validación (`Base_Maestra_2023_2025.xlsx`)
**Curso:** MCDI500 — Programación para la Ciencia de Datos (Grupo 3)  
**Docente:** Dr. Omar Iván Salinas Silva  
**Integrantes:** Madeleine Aguilera, Victoria Álvarez, Ignacio Peña y Raúl Hormazábal

---
## 1. Configuración del Entorno e Ingesta Segura desde el Paquete Compartido `src/`
Se importa el paquete compartido `src/` desde la raíz del repositorio y se carga la base maestra trianual (`118.280 filas × 21 columnas`).


In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent.parent)
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from src import (
    COLUMNAS_ESPERADAS_RETC,
    cargar_datos_seguro,
    diagnosticar_variables,
    depurar_registros_base,
    estandarizar_regiones_y_peligrosidad,
    rescatar_coordenadas_chile,
    agregar_atributos_deterministas,
    ImputadorPorGrupo,
    CodificadorCategorias,
    EscaladorRobusto,
)

RUTA_RAW = Path("F1/data/raw/Base_Maestra_2023_2025.xlsx")
df_raw = cargar_datos_seguro(RUTA_RAW, columnas_esperadas=COLUMNAS_ESPERADAS_RETC)
print(f"[OK] Dataset original cargado: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas")


[OK] Dataset original cargado: 118,280 filas × 21 columnas


## 2. Depuración Base y Diagnóstico Cuantitativo (Sin Imputar la Masa Declarada)
Siguiendo el criterio analítico del proyecto, `depurar_registros_base()` elimina las `118` filas duplicadas exactas y descarta las `21` filas que carecen de masa declarada (`cantidad_kg` nulo), ya que `cantidad_kg` es la variable central del estudio y no debe inventarse por imputación. Quedan **`118.141` declaraciones válidas**.


In [2]:
df_depurado = depurar_registros_base(df_raw)
print(f"Filas iniciales: {len(df_raw):,} -> Tras eliminar 118 duplicados y 21 sin cantidad_kg: {len(df_depurado):,}")
display(diagnosticar_variables(df_depurado))


Filas iniciales: 118,280 -> Tras eliminar 118 duplicados y 21 sin cantidad_kg: 118,141


,variable,dtype,unicos,nulos_n,nulos_pct,media,mediana,ric,outliers_iqr_n,outliers_iqr_pct
0,año,int64,3,0,0.00,2.024020e+03,2024.00,2.000000e+00,0,0.00
1,id_vu,int64,12189,0,0.00,3.923075e+06,5453607.00,5.040022e+06,0,0.00
2,rut_razon_social,str,6777,0,0.00,NaN,NaN,NaN,0,0.00
3,id_ciiu4,str,546,1,0.00,NaN,NaN,NaN,0,0.00
4,ciiu4,str,335,1,0.00,NaN,NaN,NaN,0,0.00
5,id_ciiu6,str,388,1,0.00,NaN,NaN,NaN,0,0.00
6,ciiu6,str,388,1,0.00,NaN,NaN,NaN,0,0.00
7,id_rubro,int64,20,0,0.00,1.377000e+01,17.00,1.000000e+01,0,0.00
8,rubro,str,19,1,0.00,NaN,NaN,NaN,0,0.00
9,codigo_territorial,int64,380,0,0.00,9.128160e+03,9202.00,7.011000e+03,0,0.00


## 3. Homologación Interanual (`region`, `id_ciiu4`, `peligrosidad`) y Rescate Geográfico de 2024
Se unifican las `19` etiquetas regionales en las `16` regiones oficiales de Chile, se trunca el dígito extra de `id_ciiu4` en 2024, se reducen las `812` variantes léxicas de `peligrosidad` a `93` categorías canónicas y se recupera el punto decimal de las coordenadas geográficas de 2024 dentro de los límites de Chile Continental e Insular.


In [3]:
df_homologado = estandarizar_regiones_y_peligrosidad(df_depurado, usar_optimizado=True)
df_rescatado = rescatar_coordenadas_chile(df_homologado, usar_vectorizado=True)

print(f"Regiones antes: {df_depurado['region'].nunique()} -> después: {df_rescatado['region'].nunique()}")
print(f"Glosas peligrosidad antes: {df_depurado['peligrosidad'].nunique()} -> después: {df_rescatado['peligrosidad'].nunique()}")
print(f"Nulos post-rescate en latitud: {df_rescatado['latitud'].isna().sum()} | en longitud: {df_rescatado['longitud'].isna().sum()}")


Regiones antes: 19 -> después: 16
Glosas peligrosidad antes: 812 -> después: 93
Nulos post-rescate en latitud: 279 | en longitud: 246


## 4. Atributos Deterministas y Cálculo de Persistencia Longitudinal por Establecimiento (`id_vu`)
Se calculan los indicadores directamente vinculados a la pregunta de investigación:
- `n_anios_declarados_vu` (`1`, `2` o `3` años presentes en el trienio por `id_vu`) y `es_generador_persistente_bin` (`1` si declara en 2023, 2024 y 2025).
- `es_gran_generador_bin` ($\ge 12.000\text{ kg/año}$, umbral del Art. 25 del D.S. N° 148/2003), `n_peligrosidades`, `multi_peligrosidad_bin` y `anio_indice` (habiendo eliminado las transformaciones sin justificación analítica como seno/coseno del año y largos de texto).


In [4]:
df_atributos = agregar_atributos_deterministas(df_rescatado)
resumen_persistencia = (
    df_atributos.groupby("n_anios_declarados_vu")
    .agg(
        declaraciones=("id_vu", "count"),
        establecimientos_unicos=("id_vu", "nunique"),
        masa_total_ton=("cantidad_kg", lambda s: round(s.sum() / 1000.0, 1)),
    )
)
display(resumen_persistencia)


,declaraciones,establecimientos_unicos,masa_total_ton
n_anios_declarados_vu,,,
1,6843,3064,27953.3
2,12748,2340,56136.5
3,98550,6785,1857961.4


## 5. Imputación por Grupo (`region` e `id_rubro`), Codificación Unificada y Escalamiento Robusto (`RobustScaler`)
Se aplican las transformaciones con estado sobre las `118.141` declaraciones:
1. **Imputación por grupo con bandera `_imputado`:** `latitud` y `longitud` se imputan con la mediana de su `region`, mientras que `id_lista_a` (`4.740` nulos en 2023–2024) y variables CIIU (`1` nulo) se imputan con la moda de su `id_rubro`.
2. **Codificación por rol analítico (sin contradicciones):** `rut_razon_social`, `id_ciiu4` e `id_lista_a` se codifican por frecuencia (anonimizando los RUT), y `region` (16), `estado_materia` (4) y `comuna_agrupada` (11) mediante One-Hot (sin crear `estado_materia_ord`).
3. **Escalamiento Robusto:** Se escalan las variables numéricas continuas y de conteo mediante mediana y rango intercuartílico ($\text{RIC}$), descartando las columnas de texto ya representadas numéricamente.


In [5]:
imputador = ImputadorPorGrupo(
    cols_numericas=["latitud", "longitud"],
    cols_categoricas=["id_lista_a", "id_ciiu4", "ciiu4", "id_ciiu6", "ciiu6", "rubro"],
)
codificador = CodificadorCategorias(top_n_comunas=10)
escalador = EscaladorRobusto(
    columnas=["cantidad_kg", "latitud", "longitud", "frecuencia_declaraciones_rut", "n_peligrosidades"],
    descartar_texto=True,
)

df_imputado = imputador.ajustar_transformar(df_atributos)
df_codificado = codificador.ajustar_transformar(df_imputado)
df_final = escalador.ajustar_transformar(df_codificado)

print(f"[OK] Matriz procesada de Fase 2: {df_final.shape[0]:,} filas × {df_final.shape[1]} columnas")
display(df_final.head(5))


[OK] Matriz procesada de Fase 2: 118,141 filas × 61 columnas


,año,id_vu,id_rubro,codigo_territorial,latitud,longitud,cantidad_kg,id_estado_materia,n_anios_declarados_vu,es_generador_persistente_bin,...,comuna_top_Puerto Montt,comuna_top_Quilicura,comuna_top_San Bernardo,comuna_top_Talcahuano,comuna_top_Viña del Mar,cantidad_kg_rob,latitud_rob,longitud_rob,frecuencia_declaraciones_rut_rob,n_peligrosidades_rob
0,2023,5,10,15101,-18.473556,-70.317359,24750.0,2,3,1,...,0,0,0,0,0,15.664537,3.853022,0.483897,-0.321429,0.0
1,2023,6,8,15101,-18.466515,-70.295403,15.0,1,3,1,...,0,0,0,0,0,-0.140575,3.854815,0.495950,-0.228571,1.0
2,2023,6,8,15101,-18.466515,-70.295403,28.0,1,3,1,...,0,0,0,0,0,-0.132268,3.854815,0.495950,-0.228571,0.0
3,2023,6,8,15101,-18.466515,-70.295403,28.0,1,3,1,...,0,0,0,0,0,-0.132268,3.854815,0.495950,-0.228571,1.0
4,2023,6,8,15101,-18.466515,-70.295403,80.0,1,3,1,...,0,0,0,0,0,-0.099042,3.854815,0.495950,-0.228571,1.0


## 6. Validación de Integridad (`assert`) y Exportación de `base_maestra_2023_2025_clean.csv`
Se comprueba que la matriz final sea 100% numérica, tenga `0` valores nulos y `118.141 filas × 60 columnas`, exportándola en `F2/data/processed/base_maestra_2023_2025_clean.csv`



In [6]:
assert df_final.shape == (118141, 61), f"Dimensiones inesperadas: {df_final.shape}"
assert int(df_final.isna().sum().sum()) == 0, "Existen valores nulos remanentes"
assert len(df_final.select_dtypes(exclude=["number"]).columns) == 0, "Quedaron columnas de texto"

RUTA_CLEAN_F2 = Path("F2/data/processed/base_maestra_2023_2025_clean.csv")
RUTA_CLEAN_F2.parent.mkdir(parents=True, exist_ok=True)
df_final.to_csv(RUTA_CLEAN_F2, index=False, encoding="utf-8")
print(f"[OK] Archivo exportado exitosamente en: {RUTA_CLEAN_F2} ({df_final.shape[0]:,} filas × {df_final.shape[1]} columnas)")


[OK] Archivo exportado exitosamente en: F2\data\processed\base_maestra_2023_2025_clean.csv (118,141 filas × 61 columnas)
